In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.manifold import TSNE

import os
from PIL import Image
import matplotlib.pyplot as plt

import random
import seaborn as sns

In [ ]:
import matplotlib as mpl
from matplotlib import font_manager
# Custom font path.
font_path = '../../data/Arial.ttf'

# Add the font to matplotlib.
font_manager.fontManager.addfont(font_path)

# Get the registered font name.
custom_font = font_manager.FontProperties(fname=font_path)
font_name = custom_font.get_name()
# Set the global font.
mpl.rcParams['font.family'] = font_name

In [ ]:
rs_spatial_features_file = "../../data/features/Raw/US/LosAngeles/RS/Mocov3VITB-spatial-US-LosAngeles-ep49.h5"
rs_self_features_file = "../../data/features/Raw/US/LosAngeles/RS/Mocov3VITB-self-US-LosAngeles-ep49.h5"
sv_spatial_features_file = "../../data/features/Raw/US/LosAngeles/SV/Mocov3VITB-spatial-US-LosAngeles-ep99.h5"
sv_self_features_file = "../../data/features/Raw/US/LosAngeles/SV/Mocov3VITB-self-US-LosAngeles-ep99.h5"
rs_path_file = "../../data/datasets/rs_paths/US-LosAngeles.pkl"
sv_path_file = "../../data/datasets/sv_paths/US-LosAngeles.pkl"

In [ ]:
sv_path = pd.read_pickle(sv_path_file)
sv_path['basename'] = sv_path['path'].apply(lambda x: os.path.basename(x))
sv_path

In [ ]:
rs_path = pd.read_pickle(rs_path_file)
rs_path['basename'] = rs_path['path'].apply(lambda x: '/'.join(x.split('/')[-3:]))
rs_path

In [ ]:
random.seed(42)
sampled_geoid = random.sample(list(set(sv_path['GEOID'].unique()) & set(rs_path['GEOID'].unique())), 20)
# sampled_geoid

In [ ]:
sv_path_sampled = sv_path[sv_path['GEOID'].isin(sampled_geoid)]
rs_path_sampled = rs_path[rs_path['GEOID'].isin(sampled_geoid)]
sv_path_sampled.shape, rs_path_sampled.shape

In [ ]:
os.path.exists(rs_spatial_features_file), os.path.exists(rs_self_features_file), os.path.exists(sv_spatial_features_file), os.path.exists(sv_self_features_file)

In [ ]:
sv_spatial_features = pd.read_hdf(sv_spatial_features_file, key='data')
sv_spatial_features['index'] = sv_spatial_features['index'].apply(lambda x: x.split('/')[-1])
sv_spatial_features = sv_spatial_features.merge(sv_path[['basename', 'GEOID']], left_on='index', right_on='basename')

sv_self_features = pd.read_hdf(sv_self_features_file, key='data')
sv_self_features['index'] = sv_self_features['index'].apply(lambda x: x.split('/')[-1])
sv_self_features = sv_self_features.merge(sv_path[['basename', 'GEOID']], left_on='index', right_on='basename')

rs_spatial_features = pd.read_hdf(rs_spatial_features_file, key='data')
rs_spatial_features['index'] = rs_spatial_features['index'].apply(lambda x: '/'.join(x.split('/')[-3:]))
rs_spatial_features = rs_spatial_features.merge(rs_path[['basename', 'GEOID']], left_on='index', right_on='basename')

rs_self_features = pd.read_hdf(rs_self_features_file, key='data')
rs_self_features['index'] = rs_self_features['index'].apply(lambda x: '/'.join(x.split('/')[-3:]))
rs_self_features = rs_self_features.merge(rs_path[['basename', 'GEOID']], left_on='index', right_on='basename')

In [ ]:
sv_spatial_features_sampled = sv_spatial_features[sv_spatial_features['GEOID'].isin(sampled_geoid)]
sv_self_features_sampled = sv_self_features[sv_self_features['GEOID'].isin(sampled_geoid)]
rs_spatial_features_sampled = rs_spatial_features[rs_spatial_features['GEOID'].isin(sampled_geoid)]
rs_self_features_sampled = rs_self_features[rs_self_features['GEOID'].isin(sampled_geoid)]
sv_spatial_features_sampled.shape, sv_self_features_sampled.shape, rs_spatial_features_sampled.shape, rs_self_features_sampled.shape

In [ ]:
sv_spatial_features_sampled = sv_spatial_features_sampled[['GEOID'] + list(range(768))]
sv_spatial_features_sampled.set_index('GEOID', inplace=True)
sv_self_features_sampled = sv_self_features_sampled[['GEOID'] + list(range(768))]
sv_self_features_sampled.set_index('GEOID', inplace=True)

rs_spatial_features_sampled = rs_spatial_features_sampled[['GEOID'] + list(range(768))]
rs_spatial_features_sampled.set_index('GEOID', inplace=True)
rs_self_features_sampled = rs_self_features_sampled[['GEOID'] + list(range(768))]
rs_self_features_sampled.set_index('GEOID', inplace=True)

In [ ]:
def tsne_features(features):
    tsne = TSNE(n_components=2, random_state=42)
    features_tsne = tsne.fit_transform(features)
    return features_tsne

In [ ]:
sv_spatial_tsne = tsne_features(sv_spatial_features_sampled.values)
sv_self_tsne = tsne_features(sv_self_features_sampled.values)

rs_spatial_tsne = tsne_features(rs_spatial_features_sampled.values)
rs_self_tsne = tsne_features(rs_self_features_sampled.values)

In [ ]:
sv_spatial_tsne.shape, sv_self_tsne.shape, rs_spatial_tsne.shape, rs_self_tsne.shape

In [ ]:
# save features
np.save('../../data/processed/fig/fig2_sv_spatial_tsne.npy', sv_spatial_tsne)
np.save('../../data/processed/fig/fig2_sv_self_tsne.npy', sv_self_tsne)
np.save('../../data/processed/fig/fig2_rs_spatial_tsne.npy', rs_spatial_tsne)
np.save('../../data/processed/fig/fig2_rs_self_tsne.npy', rs_self_tsne)

In [ ]:
# titles = ['RS-Spatial', 'RS-Self', 'ImageNet-Self']
titles = ['SV-Spatial', 'SV-Self']

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for i, (features_tsne, features, title) in enumerate(zip([sv_spatial_tsne, sv_self_tsne], [sv_spatial_features_sampled, sv_self_features_sampled], titles)):
    ax = axes[i]
    sns.scatterplot(x=features_tsne[:, 0], y=features_tsne[:, 1], hue=features.index, ax=ax, palette='tab20')
    ax.set_title(title, fontsize=20)
    # close legend
    ax.get_legend().remove()
    # close frame
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    # close ticks
    ax.set_xticks([])
    ax.set_yticks([])
plt.savefig('../../data/figure_assets/fig2_sv_tsne.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
titles = ['RS-Spatial', 'RS-Self']

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for i, (features_tsne, features, title) in enumerate(zip([rs_spatial_tsne, rs_self_tsne], [rs_spatial_features_sampled, rs_self_features_sampled], titles)):
    ax = axes[i]
    sns.scatterplot(x=features_tsne[:, 0], y=features_tsne[:, 1], hue=features.index, ax=ax, palette='tab20', s=200)
    ax.set_title(title, fontsize=20)
    # close legend
    ax.get_legend().remove()
    # close frame
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    # close ticks
    ax.set_xticks([])
    ax.set_yticks([])
plt.savefig('../../data/figure_assets/fig2_rs_tsne.svg', dpi=300, bbox_inches='tight')
plt.show()